In [1]:
!pip install sentence-transformers

  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)


In [2]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.5 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 9.5 MB/s  0:00:00


In [4]:
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
import torch

# ==========================================
# 1. ตั้งค่าไฟล์
# ==========================================
MODEL_PATH = "severity_model_lgb.pkl"   # ไฟล์โมเดลที่เทรนเสร็จแล้ว
NEW_DATA_PATH = "traffy_fondue_bangkok_processed.csv"
OUTPUT_PATH = "new_data_predicted.csv"  # ไฟล์ผลลัพธ์ที่จะบันทึก

# ==========================================
# 2. โหลดโมเดลและ Embedder
# ==========================================
print(f"Loading model from {MODEL_PATH}...")
bundle = joblib.load(MODEL_PATH)

regressor = bundle["regressor"]
embed_model_name = bundle["embed_model_name"]

# ตรวจสอบ Device (ใช้ GPU ถ้ามี เพื่อความเร็ว)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Embedder: {embed_model_name} on {device}...")
embedder = SentenceTransformer(embed_model_name, device=device)

# ==========================================
# 3. โหลดข้อมูลใหม่
# ==========================================
print(f"Reading data from {NEW_DATA_PATH}...")
try:
    df_new = pd.read_csv(NEW_DATA_PATH)
except Exception as e:
    print(f"Error reading file: {e}")
    exit()

# ตรวจสอบว่ามีคอลัมน์ข้อความไหม (แก้ชื่อคอลัมน์ให้ตรงกับไฟล์จริงของคุณ)
text_column = "comment" 
if text_column not in df_new.columns:
    # ลองหาชื่ออื่นเผื่อไฟล์ใหม่ใช้ชื่อไม่เหมือนเดิม
    if "comment_clean" in df_new.columns:
        text_column = "comment_clean"
    elif "text" in df_new.columns:
        text_column = "text"
    else:
        raise ValueError(f"ไม่พบคอลัมน์ข้อความ (comment) ในไฟล์ {NEW_DATA_PATH}")

print(f"Using column '{text_column}' for prediction.")

# ==========================================
# 4. แปลงข้อความและทำนาย (Inference)
# ==========================================
# เตรียมข้อมูล (จัดการค่าว่าง)
X_text = df_new[text_column].fillna("").astype(str).tolist()

print("Embedding texts... (This might take a while)")
# แปลงข้อความเป็นตัวเลข (Vector)
X_emb = embedder.encode(X_text, batch_size=64, show_progress_bar=True)

print("Predicting severity scores...")
# ให้ LightGBM ทำนาย
pred_scores = regressor.predict(X_emb)

# บังคับค่าให้อยู่ในช่วง 0-100
df_new["predicted_severity"] = np.clip(pred_scores, 0, 100)

# ==========================================
# 5. บันทึกผลลัพธ์
# ==========================================
# เลือกเซฟเฉพาะคอลัมน์ที่จำเป็น หรือเซฟทั้งหมดก็ได้
# ตัวอย่าง: เซฟ ticket_id (ถ้ามี), comment, และคะแนนที่ทำนายได้
columns_to_save = [col for col in ["ticket_id", text_column, "predicted_severity"] if col in df_new.columns]

# หรือถ้าอยากเซฟทั้งหมด:
# output_df = df_new 

output_df = df_new
output_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("------------------------------------------------")
print(f"✅ Prediction Done! Saved to: {OUTPUT_PATH}")
print("------------------------------------------------")
print(output_df.head())

Loading model from severity_model_lgb.pkl...
Loading Embedder: sentence-transformers/all-MiniLM-L6-v2 on cpu...
Reading data from traffy_fondue_bangkok_processed.csv...
Using column 'comment_clean' for prediction.
Embedding texts... (This might take a while)


Batches: 100%|██████████| 30/30 [00:06<00:00,  4.30it/s]

Predicting severity scores...
------------------------------------------------
✅ Prediction Done! Saved to: new_data_predicted.csv
------------------------------------------------
     ticket_id                                            address  \
0  2025-XU8LAE                    GCQG+P5 กรุงเทพมหานคร ประเทศไทย   
1  2025-XHLUTV  29 ซอย เทียนทะเล แขวงท่าข้าม เขตบางขุนเทียน กร...   
2  2025-J2PUCL  92/8 ซอย เทียนทะเล แขวงท่าข้าม เขตบางขุนเทียน ...   
3  2025-ZCFLVB  76/7 ถ. บางกระดี่ แขวงแสมดำ เขตบางขุนเทียน กรุ...   
4       PUNHKV  581/44 หมู่บ้านพฤกษาวิลล์ 107/5 แขวงทุ่งครุ เข...   

      district       lat        lon  DaysActive_Pending  \
0  บางขุนเทียน  13.53932  100.42544                   1   
1  บางขุนเทียน  13.54643  100.42282                   1   
2  บางขุนเทียน  13.57828  100.42066                   3   
3  บางขุนเทียน  13.59871  100.40514                   1   
4      ทุ่งครุ  13.60585  100.51859                   1   

                                       comment_cle


e:\2_1\ds\anaconda3\envs\dsde\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
